In [ ]:
%pip install semantic-link-labs

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import notebookutils 
import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.admin as admin
import sempy_labs.graph as graph
import sempy_labs.notebook as sempy_notebooks
import sempy_labs.variable_library as var_lib
import pandas as pd
import numpy as np
import requests
import os

In [ ]:
workspace_name = fabric.resolve_workspace_name()

# Instructions

Normally this module would be split across many smaller notebooks, and there would be an "installer" notebook to get the files all loaded up into your workspaces with neat folders. 

For the purposes of the 2026 Fabric Semantic Link Developer Experience Challenge, the multi-notebook module has been consolidated down to one file!

If you would like the neat multi-part solution, with abstracted functions, auto-installation from Github, and solution architecture digrams, refer to [Argel-Tal/fabric-configuration-management](https://github.com/Argel-Tal/fabric-configuration-management)

## Automatic bits
### INSTRUCTIONS: Placeholder variables

**READ THIS**

In the below code you will find placeholder values indicated by `[...]`, you will need to replace these (including the `[` `]` symbols) with values from your own tenant.

#### Managed Private Endpoints

These allow your workspace to communicate out to Private Network protected resources. You will need to establish these in all your Workspaces _(dev, uat, stg, prd, ...)_


The below code will request that resources be created for the current workspace, and the Azure Resource owner will need to approve the MPE's under the resource's networking menu. 

For more info see [Microsoft's documentation](https://learn.microsoft.com/en-au/fabric/security/security-managed-private-endpoints-create)

In [ ]:
## REPLACE THE PLACEHOLDER VALUES <...>

# create the storage container MPE
labs.managed_private_endpoint.create_managed_private_endpoint(
    (workspace_name + "-adlg2"), 
    [ADLG2 container resource ID], # TODO:replace this
    "dfs",
    ("Managed Private Endpoint for Fabric workspace: " + workspace_name)
)

# create the keyvault MPE
labs.managed_private_endpoint.create_managed_private_endpoint(
    (workspace_name + "-kv"), 
    [Key Vault resource ID], # TODO: replace this
    "vault",
    ("Managed Private Endpoint for Fabric workspace: " + workspace_name)
)

## Manual bits
### Environment

These notebooks rely on the [Semantic Link Labs package](https://semantic-link-labs.readthedocs.io/en/stable/index.html) ([Github](https://github.com/microsoft/semantic-link-labs/tree/main)). This notebook uses `%pip install` to load it, but when running in Pipelines and such you will need to make a Workspace Environment. Currently SLL does not support adding packages to an Environment (the API does). 

Environments are created as workspace artifacts, and then attached to each notebook using metadata, or the bar along the top in the notebook editor.

**Required libraries:**

Library | Version
--------|--------
Fabric  | 3.2.2
semantic-link-labs | 0.13.

### Variables

In this module, there is an assumed Azure Data Lake Gen2 enabled Azure Storage Account. 

Within this Account, it assumes a Container holding configuration data, and also another container with a subset of workspaces & domains used for testing. 

The two containers enables you to run these modules against a small list of workspaces _(files in the dev subdirectory)_, refine these modules, add to them, ... while running them in a limited blast-radius mode. Then when you're ready, you can push your content to a Production workspace, update the active variable set using a Fabric Deployment Pipeline, and all runs in the Production Workspace will automatically reference the longer list, and affect those Workspaces.

I used the names: 
- Production: `fabric-management`
- Testing: `dev-fabric-management`

Other useful variables are also stored in a variable library as follows:


name | description | value type | variable value  | Alternate value _(`Dev` workspace, if blank use default)_
:---:|-------------|------------|-----------------|-------
tenant_id | ID of your tenancy | GUID | [YOUR TENANT ID GOES HERE] |
common_capacity | ID of main Fabric capacity (assumes there's a default you assign to if assigning) | GUID | [YOUR PRIMARY FABRIC CAPACITY ID GOES HERE] |
fabric_admins | security group containing fabric admins | String | [YOUR FABRIC ADMINS GROUP ID GOES HERE] |
admin_storage_account | ADLG2 Storage Account name, which contains the configuration files | String | [YOUR ADMIN STORAGE ACCOUNT NAME GOES HERE] | 
admin_filepath | path inside the container where configuration files are stored | String | `fabric-management` | `dev-fabric-management`
admin_sp_app_id | App Reg ID of Service principal which can call Fabric Admin APIs" | GUID | [YOUR ADMIN SERVICE PRINCIPAL APP ID GOES HERE] |
keyvault_uri | URI of the Key Vault | String | `https://[YOUR KEY VAULT NAME].vault.azure.net/` | 
orchestration_sp_app_id | App Reg ID of Service principal which has access to data and is used to run workspace content | GUID | [YOUR ORCHESTRATION SERVICE PRINCIPAL APP ID GOES HERE]
orchestration_sp_app_object_id | Object ID of Service principal which has access to data and is used to run workspace content - needed for assignments | GUID | [YOUR ORCHESTRATION SERVICE PRINCIPAL OBJECT ID GOES HERE]


# Actual notebook code
To make these snippets work, you will need to have set up the Container, the service principals, the keyvaults, the tenant settings, and the variable library. Again, if you want a neat set up for this with different components isolated into different notebooks for maintainability, refer to [Argel-Tal/fabric-configuration-management](https://github.com/Argel-Tal/fabric-configuration-management)

Stuff you will need: 

Application | Entity | Notes
:----------:|--------|---------
Azure DevOps | Configuration file `.csv` | master copy
Azure Storage Account | Configuration file `.csv` | Created / Updated by DevOps pipeline
EntraID | Security group for Service Principals allowed to call EDIT Fabric Admin APIs | 
EntraID| Service Principal with <br>- Workspace access <br>- access to storage containers <br>- Ability to call public Fabric APIs | Provides the ability to run notebooks without being tied to a user identity
Azure Keyvault | Service Principal secrets  | Accessed via Keyvault queries within Fabric Notebooks
Microsoft Fabric | Fabric Capacity | 
Microsoft Fabric | Workspace(s) bound to Fabric Capacity | Also connected to Git & Deployment pipelines for CI/CD

Stuff that is nice to have: 
Application | Entity | Notes
:----------:|--------|---------
Azure DevOps    | Pipeline | **Trigger:** Changes to configuration file subdirectory on Main branch _(via Pull Request Completion)_<br>**Action:** upload (override) Configuration files into an Azure Storage Account<br>**Action:** trigger notebook execution
Azure Storage Account | Approved Private Endpoint | Created by the above SLL snippet (fill in the `[...]` placeholders) - needed if you have public access disabled for this resource
Azure Keyvault  | Approved Private Endpoint | Created by the above SLL snippet (fill in the `[...]` placeholders) - needed if you have public access disabled for this resource